In [1]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [2]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

E0000 00:00:1748117286.354714   95987 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748117286.359419   95987 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748117286.371911   95987 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748117286.371928   95987 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748117286.371929   95987 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748117286.371931   95987 computation_placer.cc:177] computation placer already registered. Please check linka

In [3]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

In [4]:
input_shape = (32, 32, 3)

efficientnetb3_model = keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        efficientnetb3_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

I0000 00:00:1748117289.854091   95987 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7258 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:01:00.0, compute capability: 6.1


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 1536)           │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       786,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 127)            │        65,151 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 127)            │           508 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 127)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         2,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,640,746 (44.41 MB)

 Trainable params: 11,552,165 (44.07 MB)

 Non-trainable params: 88,581 (346.02 KB)

In [5]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 10
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('efficientnetb3.keras')

Epoch 1/10


I0000 00:00:1748117325.309612   96026 service.cc:152] XLA service 0x76759c0405d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748117325.309629   96026 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1070, Compute Capability 6.1
I0000 00:00:1748117332.509490   96026 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/282 ━━━━━━━━━━━━━━━━━━━━ 5:51:23 75s/step - accuracy: 0.0938 - loss: 3.5339

I0000 00:00:1748117367.352791   96026 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


282/282 ━━━━━━━━━━━━━━━━━━━━ 138s 225ms/step - accuracy: 0.0984 - loss: 3.0545 - val_accuracy: 0.0840 - val_loss: 3.1382
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.3211 - loss: 2.2088 - val_accuracy: 0.0750 - val_loss: 3.8825
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.4741 - loss: 1.7055 - val_accuracy: 0.1470 - val_loss: 3.1083
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.5685 - loss: 1.3985 - val_accuracy: 0.0510 - val_loss: 5.2475
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.6347 - loss: 1.2025 - val_accuracy: 0.1050 - val_loss: 3.4093
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.6790 - loss: 1.0413 - val_accuracy: 0.2640 - val_loss: 3.2500
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.7279 - loss: 0.8993 - val_accuracy: 0.1300 - val_loss: 3.4306
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.7647 - loss: 0.7763 - val_accur

In [6]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.1319 - loss: 3.1109


[3.1232495307922363, 0.13449999690055847]